In [55]:
import numpy as np
import networkx as nx
import pandas as pd

from bokeh.plotting import figure, show
from bokeh.io import output_notebook
from bokeh.models import LinearColorMapper, ColorBar, BasicTicker, ColumnDataSource, HoverTool, LabelSet, Legend, LegendItem
from bokeh.palettes import YlOrRd9, Spectral3, RdYlGn4, Category10_3
from bokeh.transform import transform, factor_cmap
from bokeh.models import ColumnDataSource
from bokeh.layouts import row, column

from pyvis.network import Network
from IPython.display import display, HTML

from neo4j import GraphDatabase

In [28]:
df = pd.read_csv('data/grocerydb.csv')
#df.head()

## Heatmap

In [30]:
output_notebook()

stores = ['Walmart', 'Target', 'WholeFoods']

# % ultra-processed per store+category
heat = (
    df.groupby(['category', 'store'])
    .apply(lambda x: (x['FPro_class'] == 3).mean(), include_groups=False)
    .reset_index()
)
heat.columns = ['category', 'store', 'ultra_pct']

# Sort categories by average ultra_pct descending
cat_order = (
    heat.groupby('category')['ultra_pct'].mean()
    .sort_values(ascending=False)
    .index.tolist()
)

mapper = LinearColorMapper(
    palette=YlOrRd9[::-1],
    low=0, high=1
)

source = ColumnDataSource(heat)

p = figure(
    title="",
    x_range=stores,
    y_range=cat_order,
    x_axis_label="Store",
    width=650,
    height=900,
)

from bokeh.models import Title
p.add_layout(Title(
    text="FPro Class 3 share per store-category pair",
    text_font_size="10px", text_font_style="italic", align="center"
), 'above')
p.add_layout(Title(
    text="But Whole Foods Provides Less Processed Options",
    text_font_size="13px", align="center"
), 'above')
p.add_layout(Title(
    text="Ultra-Processed Foods Dominate All Three Stores",
    text_font_size="13px", text_font_style="bold", align="center"
), 'above')

p.rect(
    x='store', y='category',
    width=1, height=1,
    source=source,
    fill_color=transform('ultra_pct', mapper),
    line_color=None
)

color_bar = ColorBar(
    color_mapper=mapper,
    ticker=BasicTicker(),
    label_standoff=8,
    title="% Ultra-Processed"
)
p.add_layout(color_bar, 'right')
p.axis.axis_label_text_font_size = "11px"

show(p)


Loading BokehJS ...

## Pie Charts

In [44]:
output_notebook()

stores = ['Walmart', 'Target', 'WholeFoods']
class_labels = {
    0.0: 'Class 0: Unprocessed / Minimally Processed',
    1.0: 'Class 1: Culinary Ingredients (oils, sugar, salt)',
    2.0: 'Class 2: Processed (canned, cured, baked)',
    3.0: 'Class 3: Ultra-Processed (chips, soda, frozen meals)'
}
colors = RdYlGn4

plots = []
for store in stores:
    store_df = df[df['store'] == store]
    counts = store_df['FPro_class'].value_counts().sort_index()
    total = counts.sum()
    fracs = counts / total
    angles = fracs * 2 * np.pi
    starts = list(np.cumsum([0] + list(angles)[:-1]))
    ends = list(np.cumsum(angles))

    p = figure(
        title=store,
        width=380, height=380,
        x_range=(-2.0, 2.0), y_range=(-2.0, 2.0)
    )

    for i, k in enumerate(counts.index):
        p.wedge(
            x=0, y=0, radius=1,
            start_angle=starts[i],
            end_angle=ends[i],
            color=colors[int(k)],
            legend_label=class_labels[k],
            line_color='black'
        )
        pct = fracs[k] * 100
        mid = (starts[i] + ends[i]) / 2
        cx, cy = np.cos(mid), np.sin(mid)
        if pct >= 5:
            p.text(
                x=[0.65 * cx], y=[0.65 * cy],
                text=[f'{pct:.1f}%'],
                text_align='center', text_baseline='middle',
                text_font_size='13px', text_color='black',
                text_font_style='bold'
            )
        elif pct >= 0.5:
            p.segment(
                x0=[1.05 * cx], y0=[1.05 * cy],
                x1=[1.45 * cx], y1=[1.45 * cy],
                line_color='black', line_width=1
            )
            align = 'left' if cx > 0.1 else ('right' if cx < -0.1 else 'center')
            p.text(
                x=[1.5 * cx], y=[1.5 * cy],
                text=[f'{pct:.1f}%'],
                text_align=align, text_baseline='middle',
                text_font_size='11px', text_color='black'
            )

    p.axis.visible = False
    p.grid.visible = False
    p.legend.visible = False
    plots.append(p)

# Shared legend
legend_fig = figure(
    width=1140, height=60,
    toolbar_location=None,
    x_range=(0, 1), y_range=(0, 1)
)
for k, label in class_labels.items():
    legend_fig.scatter(marker='square', x=[-10], y=[-10], size=12,
                      color=colors[int(k)], legend_label=label)
legend_fig.legend.orientation = 'horizontal'
legend_fig.legend.location = 'center'
legend_fig.legend.label_text_font_size = '11px'
legend_fig.legend.glyph_width = 14
legend_fig.legend.glyph_height = 14
legend_fig.legend.spacing = 20
legend_fig.legend.border_line_color = None
legend_fig.axis.visible = False
legend_fig.grid.visible = False
legend_fig.outline_line_color = None

from bokeh.models import Div
from bokeh.layouts import gridplot
title_div = Div(
    text='<div style="text-align:center;font-size:16px;font-weight:bold;padding:6px 0;">Over Two-Thirds of Products Are Ultra-Processed, Even at Whole Foods</div><div style="text-align:center;font-size:11px;color:#555;font-style:italic;padding-bottom:4px;">Share of products by NOVA classification (FPro class 0\u20133) per store</div>',
    width=1140
)
show(column(title_div, gridplot([plots], toolbar_location='above', merge_tools=True), legend_fig))


Loading BokehJS ...

## Neo4j Data Loader

In [5]:
driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "password"))

def load_data(tx, row):
    tx.run("""
        MERGE (s:Store {name: $store})
        MERGE (c:Category {name: $category})
        MERGE (b:Brand {name: $brand})
        CREATE (p:Product {
            name: $name,
            fpro: $fpro,
            fpro_class: $fpro_class,
            price: $price
        })
        MERGE (p)-[:SOLD_AT]->(s)
        MERGE (p)-[:IN_CATEGORY]->(c)
        MERGE (p)-[:MADE_BY]->(b)
    """,
        store=row["store"],
        category=row["category"],
        brand=row["brand"] if row["brand"] == row["brand"] else "Unknown",
        name=row["name"],
        fpro=row["FPro"],
        fpro_class=row["FPro_class"],
        price=row["price"]
    )

with driver.session() as session:
    for _, row in df.iterrows():
        session.execute_write(load_data, row)

print(f"Loaded {len(df)} products into Neo4j")


Loaded 26250 products into Neo4j


In [72]:
print(df.groupby('FPro_class')['FPro'].agg(['min', 'max']))


                 min       max
FPro_class                    
0.0         0.000000  0.499843
1.0         0.326500  0.729093
2.0         0.294167  0.722582
3.0         0.500005  0.999056


In [71]:
output_notebook()

baking = df[df['category'] == 'baking'].copy()
baking = baking[baking['package_weight'].notna() & (baking['package_weight'] > 0)].copy()
baking['price_per_oz'] = baking['price'] / (baking['package_weight'] / 28.3495)

PRICE_SPLIT = round(baking['price_per_oz'].median(), 3)  # ~$0.250/oz
FPRO_SPLIT  = 0.5

def quadrant(row):
    cheap   = row['price_per_oz'] <= PRICE_SPLIT
    healthy = row['FPro']         <= FPRO_SPLIT
    if cheap and healthy:      return 'Budget & Less Processed'
    if not cheap and healthy:  return 'Pricier & Less Processed'
    if cheap and not healthy:  return 'Budget & Ultra-Processed'
    return                            'Pricier & Ultra-Processed'

baking['quadrant'] = baking.apply(quadrant, axis=1)

stores     = ['WholeFoods', 'Target', 'Walmart']
quad_order = [
    'Budget & Less Processed',
    'Budget & Ultra-Processed',
    'Pricier & Less Processed',
    'Pricier & Ultra-Processed',
]
quad_colors = ['#4dac26', '#d01c8b', '#f1a340', '#ca0020']

totals = baking.groupby('store').size()
counts = baking.groupby(['store', 'quadrant']).size().unstack(fill_value=0)
for q in quad_order:
    if q not in counts.columns:
        counts[q] = 0
pcts = (counts.div(totals, axis=0) * 100).round(1).reindex(stores)[quad_order]

from bokeh.models import ColumnDataSource, Title

source = ColumnDataSource(dict(stores=stores, **{q: pcts[q].tolist() for q in quad_order}))

p = figure(
    y_range=stores, width=1100, height=520,
    toolbar_location='above',
    x_axis_label='% of Baking Products',
    min_border_left=80, min_border_right=60,
    min_border_top=30, min_border_bottom=60,
)

bars = p.hbar_stack(
    stackers=quad_order, y='stores', height=0.5,
    source=source, color=quad_colors,
    line_color=None,
)

# Labels inside each segment
for i, quad in enumerate(quad_order):
    lefts  = pcts[quad_order[:i]].sum(axis=1).tolist()
    rights = pcts[quad_order[:i+1]].sum(axis=1).tolist()
    mids   = [(l + r) / 2 for l, r in zip(lefts, rights)]
    vals   = pcts[quad].tolist()
    for store, mid, val in zip(stores, mids, vals):
        if val >= 4:
            p.text(x=[mid], y=[store], text=[f'{val:.1f}%'],
                   text_align='center', text_baseline='middle',
                   text_font_size='13px', text_color='white',
                   text_font_style='bold')

p.x_range.start = 0
p.x_range.end   = 100
p.ygrid.grid_line_color = None

p.axis.axis_label_text_font_size = '13px'
p.axis.major_label_text_font_size = '12px'

p.add_layout(Title(
    text=f'Baking category only  |  Budget = ≤${PRICE_SPLIT}/oz (median price per oz)  |  FPro split: 0.5',
    text_font_size='12px', text_font_style='italic', align='center'
), 'above')

show(p)


Loading BokehJS ...